# 08 · Monitoring

**Role of this layer.** Every stage of every run (and every replay) appends one
JSON object to `runs/events.jsonl`. The log is append-only, so it doubles as an
audit trail and could be shipped to a log platform as is. The functions below
are the ones behind the Monitoring page of the app.

To make sure there is something to show, this notebook first runs all five demo
scenarios in mock mode (approving only A and B, as the CLI demo does).

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.engine import run_request
from quantlab.scenarios import SCENARIOS

for scenario in SCENARIOS:
    session = run_request(
        scenario.request, list(scenario.dataset_ids), "mock", approve=scenario.auto_approve
    )
    print(scenario.id, session.status)

A completed


B completed
C blocked
D blocked


E completed


In [3]:
from quantlab.monitoring import (
    capability_counts,
    guardrail_counts,
    kpis,
    read_events,
    recent_errors,
    stage_latency,
)

events = read_events(settings.events_path)
print(len(events), "events")
pd.Series(kpis(events))

73 events


n_runs               7.0000
n_completed          5.0000
n_blocked            2.0000
n_failed             0.0000
n_in_progress        0.0000
success_rate         0.7143
blocked_rate         0.2857
failed_rate          0.0000
n_replays            1.0000
replay_match_rate    1.0000
cache_hit_rate          NaN
dtype: float64

Latency per stage (mean and 95th percentile, milliseconds):

In [4]:
pd.DataFrame(stage_latency(events))

,stage,count,mean_ms,p95_ms
0,planning,7,0.0,0.0
1,validation,7,0.1,0.1
2,execution,5,134.8,170.2
3,narration,5,0.0,0.0


Which capabilities run most, and which guardrails fire most:

In [5]:
pd.Series(capability_counts(events), name="executions")

load_series           7
compute_returns       3
summary_stats         3
rolling_volatility    2
flag_regime           2
spread                2
Name: executions, dtype: int64

In [6]:
pd.DataFrame(guardrail_counts(events))

,code,name,severity,count
0,G12,MATERIAL_ASSUMPTION,APPROVAL,3
1,R04,ALIGNMENT_LOSS,WARN,2
2,G02,UNKNOWN_CAPABILITY,BLOCK,1
3,G13,PLANNER_REFUSED,BLOCK,1


The most recent failures (blocked runs and errors):

In [7]:
pd.DataFrame(recent_errors(events, limit=5))

,timestamp,run_id,event_type,error
0,2026-10-16T14:15:51.060Z,20261016-141551-7378b3,run_failed,The plan has blocking issues: G13:plan
1,2026-10-16T14:15:51.027Z,20261016-141550-f33382,run_failed,The plan has blocking issues: G02:s2
